In [1]:
"""
Run this to CONFIRM the missingness numbers before finalizing the recommendation.
Steps:
  1. Load train_transaction + train_identity, merge on TransactionID
  2. Compute missing % per column (this replaces the placeholder numbers in the memo)
  3. Group V-columns by matching NaN pattern
  4. Within each NaN-pattern group, drop columns highly correlated (>0.95) with another
     survivor, keeping one representative
  5. Apply the imputation rules from the recommendation table
  6. Print a summary so you can sanity check before committing

Update DATA_DIR below to point at your local copy of the CSVs.
"""

import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# 0. CONFIG
# ---------------------------------------------------------------------------
DATA_DIR = r"C:\Users\aylin\OneDrive\Desktop\INFO\INFO442\ieee-fraud-detection"
CORR_THRESHOLD = 0.95                # V-column redundancy cutoff
LOW_MISSING = 0.05                   # under this -> simple impute
HIGH_MISSING = 0.50                  # over this -> needs correlation check before drop

# ---------------------------------------------------------------------------
# 1. LOAD + MERGE
# ---------------------------------------------------------------------------
def load_data():
    train_transaction = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_identity = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    # left merge - keep all transactions even if no identity match
    df = train_transaction.merge(train_identity, on="TransactionID", how="left")
    return df


# ---------------------------------------------------------------------------
# 2. MISSINGNESS REPORT
# ---------------------------------------------------------------------------
def missingness_report(df):
    missing_pct = df.isna().mean().sort_values(ascending=False)
    report = missing_pct.to_frame("missing_pct")
    report["missing_pct"] = (report["missing_pct"] * 100).round(2)
    return report


# ---------------------------------------------------------------------------
# 3. GROUP V-COLUMNS BY NAN PATTERN, DEDUPE BY CORRELATION
# ---------------------------------------------------------------------------
def get_v_columns(df):
    return [c for c in df.columns if c.startswith("V")]


def group_by_nan_pattern(df, cols):
    """Columns that are null on the exact same rows get the same 'signature'."""
    signatures = {}
    for col in cols:
        # boolean null mask as a hashable signature (tuple is slow on 590k rows,
        # so hash the mask array bytes instead)
        sig = df[col].isna().values.tobytes()
        signatures.setdefault(sig, []).append(col)
    return list(signatures.values())  # list of column-name lists


def dedupe_correlated(df, col_group, threshold=CORR_THRESHOLD, sample_size=50000):
    """Within a NaN-pattern group, keep one column out of any highly-correlated cluster.
    Uses a random sample of rows for speed - correlation on a 50k sample is
    essentially identical to using all 590k rows, but far faster.
    """
    if len(col_group) <= 1:
        return col_group, []

    sub = df[col_group]
    if len(sub) > sample_size:
        sub = sub.sample(sample_size, random_state=42)
    sub = sub.dropna()
    if sub.shape[0] < 2 or sub.shape[1] < 2:
        return col_group, []

    corr = sub.corr().abs()
    keep = []
    drop = []
    seen = set()

    for col in col_group:
        if col in seen:
            continue
        keep.append(col)
        # find everything highly correlated with this column, mark for drop
        correlated_with_col = corr.index[corr[col] > threshold].tolist()
        for c in correlated_with_col:
            if c != col and c not in seen:
                drop.append(c)
                seen.add(c)
        seen.add(col)

    return keep, drop


def reduce_v_columns(df):
    v_cols = get_v_columns(df)
    groups = group_by_nan_pattern(df, v_cols)
    print(f"  Found {len(groups)} NaN-pattern groups among V-columns, "
          f"processing each for correlation...")

    all_keep, all_drop = [], []
    for i, group in enumerate(groups):
        if len(group) > 1:
            print(f"  Group {i+1}/{len(groups)}: {len(group)} columns...", flush=True)
        keep, drop = dedupe_correlated(df, group)
        all_keep.extend(keep)
        all_drop.extend(drop)

    print(f"V-columns: {len(v_cols)} total -> {len(all_keep)} kept, {len(all_drop)} dropped "
          f"(>{CORR_THRESHOLD} correlation within same NaN-pattern group)")
    return all_keep, all_drop


# ---------------------------------------------------------------------------
# 4. APPLY IMPUTATION RULES
# ---------------------------------------------------------------------------
def apply_null_handling(df, v_keep, v_drop):
    df = df.copy()

    # --- drop redundant V columns ---
    df = df.drop(columns=v_drop)

    # --- fill remaining V columns with flag value ---
    for col in v_keep:
        df[col] = df[col].fillna(-999)

    # --- R_emaildomain: structurally null -> binary flag, drop raw column ---
    if "R_emaildomain" in df.columns:
        df["has_recipient_email"] = df["R_emaildomain"].notna().astype(int)
        df = df.drop(columns=["R_emaildomain"])

    # --- dist1, dist2: flag value, keep numeric ---
    for col in ["dist1", "dist2"]:
        if col in df.columns:
            df[col] = df[col].fillna(-999)

    # --- card4: low missing -> "missing" category ---
    if "card4" in df.columns:
        df["card4"] = df["card4"].fillna("missing")

    # --- card1, card2, card3, card5, card6: mode impute ---
    for col in ["card1", "card2", "card3", "card5", "card6"]:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    # --- addr1, addr2: "missing" category ---
    for col in ["addr1", "addr2"]:
        if col in df.columns:
            df[col] = df[col].fillna("missing")

    # --- P_emaildomain: "missing" category ---
    if "P_emaildomain" in df.columns:
        df["P_emaildomain"] = df["P_emaildomain"].fillna("missing")

    # --- C1-C14: median impute ---
    c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
    for col in c_cols:
        df[col] = df[col].fillna(df[col].median())

    # --- D1-D15: median impute ---
    d_cols = [c for c in df.columns if c.startswith("D") and c[1:].isdigit()]
    for col in d_cols:
        df[col] = df[col].fillna(df[col].median())

    # --- M1-M9: "missing" category ---
    m_cols = [c for c in df.columns if c.startswith("M") and c[1:].isdigit()]
    for col in m_cols:
        df[col] = df[col].fillna("missing")

    # --- identity columns: presence flag + individual imputation ---
    id_numeric_cols = [c for c in df.columns if c.startswith("id_")
                        and pd.api.types.is_numeric_dtype(df[c])]
    id_cat_cols = [c for c in df.columns if c.startswith("id_")
                   and not pd.api.types.is_numeric_dtype(df[c])]

    if id_numeric_cols or id_cat_cols or "DeviceType" in df.columns:
        identity_cols = id_numeric_cols + id_cat_cols + \
            [c for c in ["DeviceType", "DeviceInfo"] if c in df.columns]
        df["has_identity_data"] = df[identity_cols].notna().any(axis=1).astype(int)

    for col in id_numeric_cols:
        df[col] = df[col].fillna(df[col].median())
    for col in id_cat_cols:
        df[col] = df[col].fillna("missing")

    # --- DeviceType: "missing" category ---
    if "DeviceType" in df.columns:
        df["DeviceType"] = df["DeviceType"].fillna("missing")

    # --- DeviceInfo: bucket rare/unseen categories into "other" ---
    if "DeviceInfo" in df.columns:
        df["DeviceInfo"] = df["DeviceInfo"].fillna("missing")
        value_counts = df["DeviceInfo"].value_counts()
        rare_threshold = 0.01 * len(df)  # values appearing in <1% of rows
        rare_values = value_counts[value_counts < rare_threshold].index
        df["DeviceInfo"] = df["DeviceInfo"].replace(rare_values, "other")

    return df


# ---------------------------------------------------------------------------
# 5. MAIN
# ---------------------------------------------------------------------------
def main():
    print("Loading data...")
    df = load_data()
    print(f"Merged shape: {df.shape}\n")

    print("Missingness report (top 20):")
    report = missingness_report(df)
    print(report.head(20))
    report.to_csv("missingness_report.csv")
    print("Full report saved to missingness_report.csv\n")

    print("Reducing V columns...")
    v_keep, v_drop = reduce_v_columns(df)

    print("\nApplying null-handling rules...")
    df_clean = apply_null_handling(df, v_keep, v_drop)

    print(f"\nFinal shape: {df_clean.shape}")
    print(f"Remaining nulls: {df_clean.isna().sum().sum()}")

    df_clean.to_csv("train_clean.csv", index=False)
    print("Saved cleaned dataset to train_clean.csv")


if __name__ == "__main__":
    main()

Loading data...
Merged shape: (590540, 434)

Missingness report (top 20):
       missing_pct
id_24        99.20
id_25        99.13
id_07        99.13
id_08        99.13
id_21        99.13
id_26        99.13
id_27        99.12
id_23        99.12
id_22        99.12
dist2        93.63
D7           93.41
id_18        92.36
D13          89.51
D14          89.47
D12          89.04
id_03        88.77
id_04        88.77
D6           87.61
id_33        87.59
id_10        87.31
Full report saved to missingness_report.csv

Reducing V columns...
  Found 15 NaN-pattern groups among V-columns, processing each for correlation...
  Group 1/15: 11 columns...
  Group 2/15: 23 columns...
  Group 3/15: 18 columns...
  Group 4/15: 22 columns...
  Group 5/15: 20 columns...
  Group 6/15: 43 columns...
  Group 7/15: 18 columns...
  Group 8/15: 11 columns...
  Group 9/15: 31 columns...
  Group 10/15: 19 columns...
  Group 11/15: 46 columns...
  Group 12/15: 16 columns...
  Group 13/15: 32 columns...
  Group 14

C:\Users\aylin\AppData\Local\Temp\ipykernel_31080\2001482772.py:137: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["has_recipient_email"] = df["R_emaildomain"].notna().astype(int)
C:\Users\aylin\AppData\Local\Temp\ipykernel_31080\2001482772.py:187: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["has_identity_data"] = df[identity_cols].notna().any(axis=1).astype(int)



Final shape: (590540, 347)
Remaining nulls: 0
Saved cleaned dataset to train_clean.csv


In [4]:
import os
print(os.getcwd())

C:\Users\aylin
